# Braintrust trace review with ReactCode and Label Studio

This notebook is the companion for the **Braintrust → Label Studio Enterprise** ReactCode integration tutorial.

You will:
- generate or pull traces from Braintrust
- normalize them into a consistent trace-review schema
- create a Label Studio project via API
- import traces as tasks
- review + annotate in a custom ReactCode UI

> **Requirement:** ReactCode is available in **Label Studio Enterprise** only.

## 1) Configuration

Create a `.env` file in the repository root (or the same directory as this notebook) with the following variables:

```bash
# Label Studio Enterprise
LABEL_STUDIO_HOST=http://localhost:8080       # or your LS Enterprise instance URL
LABEL_STUDIO_API_KEY=your_label_studio_api_key

# Braintrust
BRAINTRUST_API_KEY=your_braintrust_api_key
BRAINTRUST_PROJECT=your_project_name          # project name for tracing and fetching

# Anthropic (only needed for Section 3a sample trace generation)
ANTHROPIC_API_KEY=your_anthropic_api_key
```

In [74]:
import os
from dotenv import load_dotenv

# Load .env from current directory or repository root
# override=True ensures kernel picks up .env changes without restart
load_dotenv(override=True)
load_dotenv(os.path.join(os.path.dirname(os.getcwd()), '.env'), override=True)  # fallback: repo root

# Label Studio Enterprise
LABEL_STUDIO_HOST = os.getenv('LABEL_STUDIO_HOST', 'http://localhost:8080')
LABEL_STUDIO_API_KEY = os.getenv('LABEL_STUDIO_API_KEY', '')

# Braintrust
BRAINTRUST_API_KEY = os.getenv('BRAINTRUST_API_KEY', '')
BRAINTRUST_PROJECT = os.getenv('BRAINTRUST_PROJECT', '')

# Anthropic (only needed for sample trace generation in Section 3a)
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY', '')

print('LABEL_STUDIO_HOST:', LABEL_STUDIO_HOST)
print('BRAINTRUST_PROJECT:', BRAINTRUST_PROJECT or '(not set)')
print('Has LABEL_STUDIO_API_KEY?', bool(LABEL_STUDIO_API_KEY))
print('Has BRAINTRUST_API_KEY?', bool(BRAINTRUST_API_KEY))
print('Has ANTHROPIC_API_KEY?', bool(ANTHROPIC_API_KEY))

LABEL_STUDIO_HOST: https://app.humansignal.com
BRAINTRUST_PROJECT: ls-project
Has LABEL_STUDIO_API_KEY? True
Has BRAINTRUST_API_KEY? True
Has ANTHROPIC_API_KEY? True


## 2) Install dependencies

In [69]:
!pip -q install requests label-studio-sdk python-dotenv braintrust braintrust-api braintrust-langchain langchain langchain-anthropic anthropic langgraph


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 3) Label Studio ReactCode config

The UI loaded from `label_config.py` has three panels:

| Panel | Purpose |
|---|---|
| **Turns** (left) | Scrollable list of all turns. Filter by role, search by content. Each card shows role, tool badges, latency, and verdict once annotated. |
| **Turn Details** (center) | Full content, tool call inputs/outputs, token usage, latency, and Claude's extended thinking (when present). |
| **Annotation** (right) | Structured form for evaluating each turn — see annotation model below. |

**Annotation model — what you capture per turn:**
- **Verdict** — Pass or Fail
- **Issue tags** — taxonomy across 5 categories: Accuracy & Faithfulness, Tool & Retrieval, Reasoning & Planning, Response Quality, Safety & Compliance
- **Severity** — Critical / Major / Minor / Suggestion
- **Expected behavior** — free text: what should the agent have done instead?
- **Comments** — any additional notes

A **trace-level verdict** (Pass / Fail / Mixed) in the bottom bar captures overall conversation quality, independent of individual turn verdicts.

In [70]:
# Load 3-panel ReactCode config (label_config.py + template.js in same directory)
_config_paths = [
  'label_config.py',
  os.path.join(os.getcwd(), 'tutorials', 'how-to-review-braintrust-traces-with-label-studio', 'label_config.py'),
]
_config_path = next((p for p in _config_paths if os.path.exists(p)), None)
if _config_path:
  import importlib.util
  _spec = importlib.util.spec_from_file_location('label_config', _config_path)
  _mod = importlib.util.module_from_spec(_spec)
  _spec.loader.exec_module(_mod)
  LABEL_CONFIG_XML = _mod.LABEL_CONFIG_XML
else:
  raise FileNotFoundError('label_config.py not found. Ensure it is in the same directory as this notebook.')
print(LABEL_CONFIG_XML[:300] + '\n...')

<View>
  <ReactCode style="height: 95vh" name="trace" toName="trace" outputs='{"trace_id":"string","turn_id":"string","turn_role":"string","verdict":"string","failure_modes":"array","severity":"string","expected_behavior":"string","comments":"string"}'>
    <![CDATA[
    function TraceAnnotator({ Re
...


## 3a) Generate sample traces (optional)

If you already have traces in Braintrust, **skip this cell** — set `GENERATE_TRACES = False` and go directly to Section 4.

Otherwise, this cell creates a ReAct agent with multiple tools and runs 4 multi-turn conversations using **Claude with extended thinking** to produce realistic traces in your Braintrust project. Requires `ANTHROPIC_API_KEY`.

Extended thinking lets Claude reason through complex, ambiguous problems step-by-step before responding. The thinking content is captured in the trace and visible in the Label Studio UI.

In [50]:
GENERATE_TRACES = True  # Set to False if you already have traces in Braintrust

if GENERATE_TRACES:
    import braintrust
    from braintrust_langchain import BraintrustCallbackHandler, set_global_handler
    from langchain_core.tools import tool
    from langchain_core.messages import HumanMessage
    from langchain_anthropic import ChatAnthropic
    from langchain.agents import create_agent as create_react_agent

    if not ANTHROPIC_API_KEY:
        raise RuntimeError('ANTHROPIC_API_KEY is required for trace generation. Set it in your .env or set GENERATE_TRACES=False.')
    if not BRAINTRUST_PROJECT:
        raise RuntimeError('BRAINTRUST_PROJECT is required. Set it in your .env file.')

    # Initialize Braintrust logger + LangChain callback handler
    braintrust.init_logger(project=BRAINTRUST_PROJECT, api_key=BRAINTRUST_API_KEY)
    handler = BraintrustCallbackHandler()
    set_global_handler(handler)

    # --- Tools ---
    @tool
    def calculator(expression: str) -> str:
        """Evaluate a math expression. Input should be a valid Python math expression."""
        try:
            return str(eval(expression))
        except Exception as e:
            return f"Error: {e}"

    @tool
    def search_knowledge_base(query: str) -> str:
        """Search an internal knowledge base for information about company policies, products, or procedures."""
        kb = {
            "refund": "Refund policy: Full refund within 30 days of purchase. After 30 days, store credit only. Damaged items: full refund at any time with photo evidence.",
            "shipping": "Shipping: Standard (5-7 business days, free over $50), Express (2-3 days, $12.99), Overnight ($24.99). International shipping available to 40+ countries.",
            "warranty": "Warranty: 1-year limited warranty on all electronics. 2-year extended warranty available for $29.99. Covers manufacturing defects only.",
            "pricing": "Enterprise pricing: Base plan $99/mo (up to 10 users), Pro plan $249/mo (up to 50 users), Enterprise plan custom pricing. Annual billing saves 20%.",
        }
        query_lower = query.lower()
        results = [v for k, v in kb.items() if k in query_lower]
        return results[0] if results else f"No results found for: {query}"

    @tool
    def get_weather(city: str) -> str:
        """Get the current weather for a city."""
        weather_data = {
            "new york": "New York: 72°F, Partly Cloudy, Humidity 65%, Wind 8 mph SW",
            "london": "London: 58°F, Overcast, Humidity 80%, Wind 12 mph W",
            "tokyo": "Tokyo: 82°F, Clear, Humidity 55%, Wind 5 mph NE",
            "paris": "Paris: 63°F, Light Rain, Humidity 75%, Wind 10 mph NW",
        }
        return weather_data.get(city.lower(), f"Weather data not available for {city}")

    # Claude with extended thinking enabled.
    # Extended thinking lets Claude reason step-by-step before responding,
    # producing richer traces and surfacing the model's reasoning process.
    llm = ChatAnthropic(
        model='claude-sonnet-4-5-20250929',
        max_tokens=16000,
        thinking={'type': 'enabled', 'budget_tokens': 5000},
    )
    agent = create_react_agent(llm, [calculator, search_knowledge_base, get_weather])

    # --- 4 multi-turn conversations designed to elicit extended thinking ---
    # These scenarios involve conflicting policies, ambiguous constraints, and
    # multi-step reasoning — ideal for demonstrating Claude's thinking process.
    conversations = [
        # 1: Complex refund edge case — overlapping policies and time limits
        [
            "I bought a product 37 days ago. I know it's past your 30-day refund window, but I just discovered it has a manufacturing defect that's been slowly getting worse since day one. I also purchased the extended warranty at the time. What are all my options here?",
            "The item costs $289. If I accept store credit rather than a cash refund, can I apply that credit toward an extended warranty on a different product and still keep the original warranty claim open on this defective unit?",
        ],
        # 2: Pricing optimization — mixed tiers and forward-looking cost projection
        [
            "We have 60 employees but only 40 need full platform access. The other 20 just need read-only reporting. Can we mix subscription plans to reduce our costs? What's the most efficient setup?",
            "If we commit to annual billing today and expect to onboard 15 more full-access users next quarter, what's our projected total spend for the first 12 months?",
        ],
        # 3: Multi-city retreat planning — weather + travel logistics + client impression
        [
            "I'm organizing a 20-person client retreat and deciding between Tokyo, London, and New York. Factor in current weather conditions, travel logistics, and the fact that this is a client-facing event where neutral ground matters.",
            "Given that 12 of our attendees are based in New York and 8 are in London, and we want to minimize total travel disruption while maintaining the impression of a neutral, premium venue — re-evaluate the three options and give me your final recommendation with reasoning.",
        ],
        # 4: Escalation with multi-step financial reconciliation
        [
            "I need to escalate a complaint. I ordered 3 items totaling $180 and paid $12.99 for express shipping. Two items arrived on time but one was damaged in transit. I need the replacement urgently — standard shipping won't work for my deadline.",
            "If I return the damaged item for a full refund and pay for express shipping on the replacement, what's my net out-of-pocket cost compared to if everything had arrived correctly the first time? Walk me through the numbers.",
        ],
    ]

    for i, conv_messages in enumerate(conversations, 1):
        print(f"\n--- Conversation {i} ---")

        @braintrust.traced(name=f"conversation_{i}")
        def run_conversation(messages):
            chat_history = []
            final_reply = ""
            for msg_text in messages:
                print(f"  User: {msg_text[:80]}...")
                chat_history.append(HumanMessage(content=msg_text))
                result = agent.invoke({'messages': chat_history})
                chat_history = result['messages']
                final_reply = result['messages'][-1].content
                if isinstance(final_reply, list):
                    # Extract text from content blocks (extended thinking responses)
                    final_reply = ' '.join(
                        b.get('text', '') for b in final_reply
                        if isinstance(b, dict) and b.get('type') == 'text'
                    )
                print(f"  Assistant: {str(final_reply)[:100]}...")
            return final_reply

        run_conversation(conv_messages)

    braintrust.flush()
    print(f'\n✓ Generated {len(conversations)} traces (1 per conversation). Proceed to Section 4.')
else:
    print('Skipped trace generation. Proceed to Section 4.')


--- Conversation 1 ---
  User: I bought a product 37 days ago. I know it's past your 30-day refund window, but ...
  Assistant: Great news! Based on your situation, you have **multiple options**:

## Your Options:

**1. Manufact...
  User: The item costs $289. If I accept store credit rather than a cash refund, can I a...
  Assistant: I need to clarify something important here: **You can't have both the store credit AND keep the warr...

--- Conversation 2 ---
  User: We have 60 employees but only 40 need full platform access. The other 20 just ne...
  Assistant: Based on the information available in the knowledge base, here's what I found:

**Current Pricing St...
  User: If we commit to annual billing today and expect to onboard 15 more full-access u...
  Assistant: Unfortunately, I don't have enough information in the knowledge base to calculate your projected 12-...

--- Conversation 3 ---
  User: I'm organizing a 20-person client retreat and deciding between Tokyo, London, an...


## 4) Braintrust API client

Fetches traces (spans) from Braintrust using the REST API with BTQL queries. Spans are grouped by `root_span_id` into traces.

In [71]:
import requests
from braintrust_api import Braintrust
from typing import Any, Dict, List, Optional


class BraintrustClient:
    """Fetches traces from Braintrust via the REST API + BTQL."""

    API_URL = 'https://api.braintrust.dev'

    def __init__(self, api_key: str, project_name: str):
        self.api_key = api_key
        self.project_name = project_name
        self.headers = {
            'Authorization': f'Bearer {api_key}',
            'Content-Type': 'application/json',
        }
        # Resolve project ID from project name
        self.project_id = self._resolve_project_id()

    def _resolve_project_id(self) -> str:
        """Look up the project ID by name."""
        client = Braintrust(api_key=self.api_key)
        for project in client.projects.list():
            if project.name == self.project_name:
                return project.id
        raise ValueError(f'Project "{self.project_name}" not found in Braintrust. Check BRAINTRUST_PROJECT.')

    def _btql(self, query: str) -> List[Dict[str, Any]]:
        """Execute a BTQL query and return results."""
        r = requests.post(
            f'{self.API_URL}/btql',
            headers=self.headers,
            json={'query': query, 'fmt': 'json'},
            timeout=60,
        )
        r.raise_for_status()
        payload = r.json()
        return payload.get('data', [])

    def list_traces(self, limit: int = 20) -> List[Dict[str, Any]]:
        """Fetch recent traces (top-level spans) from the project."""
        query = f"""SELECT id, span_id, root_span_id, input, output, metadata, metrics,
                            scores, created, span_attributes, error
                     FROM project_logs('{self.project_id}')
                     WHERE span_id = root_span_id
                     ORDER BY created DESC
                     LIMIT {limit}"""
        return self._btql(query)

    def get_trace_spans(self, root_span_id: str) -> List[Dict[str, Any]]:
        """Fetch all spans belonging to a trace (by root_span_id)."""
        query = f"""SELECT id, span_id, root_span_id, span_parents, input, output,
                            metadata, metrics, scores, created, span_attributes, error
                     FROM project_logs('{self.project_id}')
                     WHERE root_span_id = '{root_span_id}'
                     ORDER BY created ASC
                     LIMIT 200"""
        return self._btql(query)


if not BRAINTRUST_API_KEY:
    raise RuntimeError('Missing BRAINTRUST_API_KEY — set it in your .env file.')
if not BRAINTRUST_PROJECT:
    raise RuntimeError('Missing BRAINTRUST_PROJECT — set it in your .env file.')

bt = BraintrustClient(BRAINTRUST_API_KEY, BRAINTRUST_PROJECT)
print(f'Braintrust client ready — project: {BRAINTRUST_PROJECT} (ID: {bt.project_id})')

Braintrust client ready — project: ls-project (ID: 5f456c27-fd45-4414-aba1-42487a82ca7f)


## 5) Normalize Braintrust traces → unified schema

Braintrust stores traces as a flat list of spans with typed `span_attributes` (`llm`, `tool`, `task`, `function`). This cell extracts the relevant spans and maps them into a flat sequence of turns — the same schema used by all three platform integrations — so the ReactCode UI doesn't need to know which platform the trace came from.

Each turn carries: `role`, `content`, `tool_name`, `tool_input`, `tool_calls`, `model`, `usage` (token counts), `duration_ms`, and `thinking` (Claude extended thinking blocks, when present).

In [72]:
import json as _json


def _to_str(x):
    """Safely convert any value to a readable string."""
    if x is None:
        return ''
    if isinstance(x, str):
        return x
    if isinstance(x, (dict, list)):
        try:
            return _json.dumps(x, indent=2, default=str)
        except Exception:
            return str(x)
    return str(x)


def _extract_content(obj):
    """Extract plain-text content from various data shapes."""
    if obj is None:
        return ''
    if isinstance(obj, str):
        return obj
    if isinstance(obj, dict):
        for key in ('content', 'text', 'input', 'output', 'result'):
            if isinstance(obj.get(key), str) and obj[key].strip():
                return obj[key]
        return _to_str(obj)
    if isinstance(obj, list):
        parts = [_extract_content(item) for item in obj if _extract_content(item).strip()]
        return '\n'.join(parts) if parts else _to_str(obj)
    return str(obj)


def _duration_ms(start_str, end_str):
    """Compute duration in milliseconds from two ISO timestamp strings."""
    if not start_str or not end_str:
        return None
    try:
        from datetime import datetime
        def _parse(s):
            return datetime.fromisoformat(str(s).replace('Z', '+00:00'))
        return int((_parse(end_str) - _parse(start_str)).total_seconds() * 1000)
    except Exception:
        return None


def _duration_ms_from_metrics(metrics):
    """Compute duration from Braintrust metrics.start / metrics.end (Unix seconds)."""
    if not isinstance(metrics, dict):
        return None
    start = metrics.get('start')
    end = metrics.get('end')
    if start is not None and end is not None:
        try:
            return int((float(end) - float(start)) * 1000)
        except Exception:
            pass
    return None


def _split_thinking(content):
    """Split Anthropic extended-thinking content blocks into (text, thinking).

    Returns (text: str, thinking: str | None).
    Plain strings pass through unchanged.
    """
    if isinstance(content, str):
        return content, None
    if isinstance(content, list):
        text_parts, thinking_parts = [], []
        for block in content:
            if isinstance(block, dict):
                if block.get('type') == 'thinking':
                    thinking_parts.append(block.get('thinking', ''))
                elif block.get('type') == 'text':
                    text_parts.append(block.get('text', ''))
                else:
                    val = block.get('text') or block.get('content') or ''
                    if val:
                        text_parts.append(str(val))
            elif isinstance(block, str):
                text_parts.append(block)
        return '\n\n'.join(text_parts), '\n\n'.join(thinking_parts) or None
    return str(content) if content else '', None


def normalize_braintrust_trace(root_span, all_spans):
    """Convert Braintrust spans into the unified trace schema.

    Braintrust span structure (from LangGraph + BraintrustCallbackHandler):
      conversation_N (root, type=function)
        └── LangGraph (type=task)
              ├── model (type=task)
              │     └── ChatOpenAI (type=llm)
              ├── tools (type=task)
              │     └── search_knowledge_base (type=tool)
              ...

    Strategy:
      1. Sort spans by timestamp
      2. type=llm → extract user messages from input + assistant response from output
      3. type=tool → extract tool execution as a tool turn
      4. Skip structural spans (type=task, type=function)
    """
    trace_id = root_span.get('span_id') or root_span.get('id', '')

    spans_sorted = sorted(all_spans, key=lambda s: s.get('created') or '')

    turns = []
    turn_counter = 0
    seen_user_messages = set()

    def add_turn(role, content, **kwargs):
        nonlocal turn_counter
        if not content or not content.strip():
            return
        turn = {
            'turn_id': f'turn_{turn_counter}',
            'role': role,
            'content': content.strip(),
            'timestamp': kwargs.get('timestamp', ''),
        }
        for k in ('model', 'usage', 'tool_calls', 'tool_name', 'tool_input', 'duration_ms', 'thinking'):
            v = kwargs.get(k)
            if v is not None:
                turn[k] = v
        turns.append(turn)
        turn_counter += 1

    for span in spans_sorted:
        attrs = span.get('span_attributes') or {}
        stype = (attrs.get('type') or '').lower()
        ts = span.get('created') or ''
        # Braintrust metrics.start/end are Unix timestamps in seconds
        duration = _duration_ms_from_metrics(span.get('metrics'))
        inp = span.get('input')
        out = span.get('output')
        span_metrics = span.get('metrics') or {}

        # ── LLM span: ChatOpenAI ──
        if stype == 'llm':
            messages = inp
            if isinstance(messages, list) and messages and isinstance(messages[0], list):
                messages = messages[0]

            if isinstance(messages, list):
                for msg in messages:
                    if isinstance(msg, dict):
                        role = msg.get('role') or msg.get('type', '')
                        if role in ('user', 'human'):
                            content = msg.get('content', '')
                            if isinstance(content, list):
                                content = ' '.join(
                                    p.get('text', '') if isinstance(p, dict) else str(p) for p in content
                                )
                            if content and content.strip():
                                msg_key = content[:200]
                                if msg_key not in seen_user_messages:
                                    seen_user_messages.add(msg_key)
                                    add_turn('user', content, timestamp=ts)

            raw_content = ''
            tool_calls = []
            if isinstance(out, dict):
                gens = out.get('generations')
                if isinstance(gens, list) and gens and isinstance(gens[0], list) and gens[0]:
                    gen = gens[0][0]
                    if isinstance(gen, dict):
                        message = gen.get('message') or gen
                        raw_content = message.get('content', '') or ''
                        ak = message.get('additional_kwargs') or {}
                        tc_raw = ak.get('tool_calls') or message.get('tool_calls') or []
                        for tc in tc_raw:
                            if isinstance(tc, dict):
                                func = tc.get('function') or {}
                                tool_calls.append({
                                    'tool_name': func.get('name') or tc.get('name', 'unknown'),
                                    'input': _to_str(func.get('arguments') or tc.get('args', '')),
                                    'call_id': tc.get('id', ''),
                                })

            assistant_content, thinking = _split_thinking(raw_content)

            model_name = attrs.get('name') or ''
            usage = None
            if span_metrics.get('prompt_tokens') or span_metrics.get('completion_tokens'):
                usage = {
                    'input_tokens': span_metrics.get('prompt_tokens', 0),
                    'output_tokens': span_metrics.get('completion_tokens', 0),
                }

            if assistant_content and assistant_content.strip():
                add_turn(
                    'assistant', assistant_content,
                    timestamp=ts, model=model_name, usage=usage,
                    tool_calls=tool_calls if tool_calls else None,
                    duration_ms=duration, thinking=thinking,
                )

        # ── Tool span ──
        elif stype == 'tool':
            tool_name = attrs.get('name') or 'unknown'
            tool_input = _to_str(inp) if inp else ''
            tool_output = (out.get('content', '') or _extract_content(out)) if isinstance(out, dict) else _extract_content(out)

            if tool_output:
                add_turn(
                    'tool', tool_output,
                    timestamp=ts, tool_name=tool_name, tool_input=tool_input,
                    duration_ms=duration,
                )

    if not turns:
        root_input = _extract_content(root_span.get('input'))
        root_output = _extract_content(root_span.get('output'))
        if root_input:
            add_turn('user', root_input, timestamp=root_span.get('created', ''))
        if root_output:
            add_turn('assistant', root_output, timestamp=root_span.get('created', ''))

    return {
        'trace_id': str(trace_id),
        'session_id': str(trace_id),
        'metadata': {
            'name': (root_span.get('span_attributes') or {}).get('name') or root_span.get('id', ''),
            'source': 'braintrust',
            'tags': root_span.get('tags') or [],
            'start_time': root_span.get('created') or '',
            'scores': root_span.get('scores') or {},
        },
        'turns': turns,
    }


print('✓ Normalization functions defined')


✓ Normalization functions defined


## 6) Fetch, normalize, and import into Label Studio

Fetches traces from Braintrust, normalizes them, creates a Label Studio project with the ReactCode config, and imports the tasks.

In [75]:
from label_studio_sdk import LabelStudio
from label_studio_sdk.core.request_options import RequestOptions
from typing import Any, Dict, List

# Extended timeout for large ReactCode label configs
_REQUEST_OPTS = RequestOptions(timeout_in_seconds=120)


def create_project(ls_host: str, api_key: str, title: str, label_config: str) -> int:
    client = LabelStudio(base_url=ls_host, api_key=api_key)
    project = client.projects.create(title=title, label_config=label_config, request_options=_REQUEST_OPTS)
    return int(project.id)


def import_tasks(ls_host: str, api_key: str, project_id: int, tasks: List[Dict[str, Any]]) -> Any:
    client = LabelStudio(base_url=ls_host, api_key=api_key)
    return client.projects.import_tasks(id=project_id, request=tasks, return_task_ids=True)


if not LABEL_STUDIO_API_KEY:
    raise RuntimeError('Missing LABEL_STUDIO_API_KEY — set it in your .env file.')

# 1) Fetch traces from Braintrust (most recent, limited to 20)
root_spans = bt.list_traces(limit=20)

if not root_spans:
    raise RuntimeError(
        'No traces returned from Braintrust. Ensure you have traces in your project '
        'and that BRAINTRUST_* credentials are correct. Run Section 3a to generate sample traces.'
    )

print(f'Fetched {len(root_spans)} root spans from Braintrust')

# 2) Normalize each trace — only include traces with child spans
tasks: List[Dict[str, Any]] = []
skipped = 0
for root in root_spans:
    root_span_id = root.get('span_id') or root.get('root_span_id')
    if not root_span_id:
        continue
    all_spans = bt.get_trace_spans(root_span_id)

    if len(all_spans) <= 1:
        skipped += 1
        continue

    normalized = normalize_braintrust_trace(root, all_spans)

    if normalized['turns']:
        tasks.append({'data': normalized})
        print(f"  + Trace {root_span_id[:12]}... -> {len(normalized['turns'])} turns "
              f"({sum(1 for t in normalized['turns'] if t['role']=='user')} user, "
              f"{sum(1 for t in normalized['turns'] if t['role']=='assistant')} assistant, "
              f"{sum(1 for t in normalized['turns'] if t['role']=='tool')} tool)")

if skipped:
    print(f'  (skipped {skipped} traces without child spans)')

print(f'\nPrepared {len(tasks)} tasks for import')

# 3) Create project and import
project_id = create_project(
    ls_host=LABEL_STUDIO_HOST,
    api_key=LABEL_STUDIO_API_KEY,
    title=f'Braintrust Trace Review ({BRAINTRUST_PROJECT})',
    label_config=LABEL_CONFIG_XML,
)
print(f'Created project: {project_id}')

resp = import_tasks(LABEL_STUDIO_HOST, LABEL_STUDIO_API_KEY, project_id, tasks)
print(f'Imported {len(tasks)} tasks')

print(f'\nDone! Open your project: {LABEL_STUDIO_HOST.rstrip("/")}/projects/{project_id}')

Fetched 4 root spans from Braintrust
  + Trace a35a87cd-39f... -> 15 turns (2 user, 6 assistant, 7 tool)
  + Trace 721cb75c-885... -> 8 turns (2 user, 3 assistant, 3 tool)
  + Trace 4b78e084-97c... -> 14 turns (2 user, 4 assistant, 8 tool)
  + Trace 48d33262-67a... -> 11 turns (2 user, 3 assistant, 6 tool)

Prepared 4 tasks for import
Created project: 239563
Imported 4 tasks

Done! Open your project: https://app.humansignal.com/projects/239563


## What's next

- **Start annotating**: Open the project link above and click through traces in the ReactCode UI
- **Share with SMEs**: Invite domain experts to your Label Studio project for collaborative evaluation
- **Incremental sync**: Re-run sections 4–6 periodically to pull new traces
- **Export annotations**: Use the Label Studio SDK or REST API to pull structured annotations for downstream analysis or fine-tuning
- **Custom taxonomy**: Edit `template.js` to add failure modes specific to your domain
- **Langfuse / LangSmith**: See companion tutorials for other observability platforms